# Site-Safe fire/smoke detector: training on Google Colab

This notebook trains the fire and smoke detector used by `ai-module/inference.py` (`models/fire_smoke.pt`, classes `0=fire`, `1=smoke`).

The validation split drives early stopping. The test split is evaluated once at the end and is never used for tuning.

**Before you start:**

- Use a dataset you are licensed to use, for example the public D-Fire dataset (YOLO format; classes 0=smoke, 1=fire). Read its licence and record its source in the report.
- Do not create real fire or smoke to collect data.
- Prefer a dataset that includes plenty of negatives: sunsets, orange vests, welding sparks, fog and dust. These are what cause false alarms on a construction site.

## 1. GPU runtime and pinned dependency
Choose **Runtime > Change runtime type > T4 GPU**. Ultralytics is pinned to the same version as the PPE notebook.

In [ ]:
%pip install -q "ultralytics==8.4.41" "PyYAML==6.0.3"

## 2. Get the repository and the dataset
Put the repository at `/content/site-safe` (clone it, or upload a ZIP of it to Drive and unzip). Put the dataset folder (with `train/images`, `train/labels`, `test/images`, `test/labels`) or a ZIP of it on Drive, then edit the paths.

In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile

from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/site-safe")
DATASET_SOURCE = Path("/content/drive/MyDrive/site-safe/D-Fire.zip")  # EDIT ME (folder or .zip)
SOURCE_CLASSES = "smoke,fire"  # class names in the dataset's id order; D-Fire is smoke,fire

assert (REPO / "ai-module" / "fire" / "prepare_fire_dataset.py").is_file(), "Put the repository at /content/site-safe first"
RAW = Path("/content/fire_raw")
if not RAW.exists():
    if DATASET_SOURCE.suffix == ".zip":
        with zipfile.ZipFile(DATASET_SOURCE) as z:
            z.extractall(RAW)
    else:
        shutil.copytree(DATASET_SOURCE, RAW)
# Use the folder that contains train/ (a ZIP may add one level).
candidates = [p for p in [RAW, *RAW.iterdir()] if (p / "train" / "images").is_dir()]
assert candidates, f"No train/images under {RAW}"
RAW_ROOT = candidates[0]
print("dataset root:", RAW_ROOT)

## 3. Validate, remap classes and build the split
`prepare_fire_dataset.py` remaps the labels to Site-Safe's class order. It carves a validation split from train that keeps frames from the same clip together, keeps negatives, and leaves out any image whose labels are invalid. Read the counts and the skipped images before training.

In [ ]:
import json
OUT = Path("/content/fire_dataset_v1")
subprocess.run(["python", str(REPO / "ai-module/fire/prepare_fire_dataset.py"),
                "--source", str(RAW_ROOT), "--source-classes", SOURCE_CLASSES, "--out", str(OUT)], check=True)
manifest = json.loads((OUT / "manifest.json").read_text())
print(json.dumps(manifest["counts"], indent=2))
print("skipped images:", len(manifest["skipped_images"]))
DATA_YAML = OUT / "data.yaml"

## 4. Train
YOLO11s gives a reasonable speed and accuracy balance for a secondary detector that runs on every fifth frame. Settings are recorded in the run directory.

In [ ]:
import torch, random, numpy as np
from ultralytics import YOLO
assert torch.cuda.is_available(), "Select a GPU runtime"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

RUN_NAME = "fire_smoke_v1"
TRAIN_ARGS = dict(data=str(DATA_YAML), epochs=100, imgsz=640, batch=16, patience=20, optimizer="AdamW",
                  lr0=0.001, seed=SEED, deterministic=True, device=0, workers=2,
                  project="/content/runs/train", name=RUN_NAME, exist_ok=False, plots=True)
model = YOLO("yolo11s.pt")
model.train(**TRAIN_ARGS)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.is_file()

## 5. Held-out test evaluation (run once)

In [ ]:
metrics = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=640, batch=16, device=0,
                                project="/content/runs/test", name=f"{RUN_NAME}_held_out", exist_ok=False)
names = metrics.names
results = {
    "overall": {"precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
                "map50": float(metrics.box.map50), "map50_95": float(metrics.box.map)},
    "per_class": {names[int(c)]: {"precision": float(p), "recall": float(r), "map50": float(m50), "map50_95": float(m)}
                  for c, p, r, m50, m in zip(metrics.box.ap_class_index, metrics.box.p, metrics.box.r,
                                             metrics.box.ap50, metrics.box.ap)},
    "test_counts": manifest["counts"].get("test", {}),
}
print(json.dumps(results, indent=2))

## 6. Export
Copy `fire_smoke.pt` to `ai-module/models/fire_smoke.pt` on the demo machine. Keep the evidence files with it and record the checksum in the report.

Image-level mAP is not the same as alert quality. Also measure alerts on recorded clips, including hard negatives, before claiming results.

In [ ]:
import hashlib
EXPORT = Path("/content/artifacts") / RUN_NAME
EXPORT.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, EXPORT / "fire_smoke.pt")
shutil.copy2(OUT / "manifest.json", EXPORT / "dataset_manifest.json")
shutil.copy2(RUN_DIR / "args.yaml", EXPORT / "args.yaml")
(EXPORT / "held_out_test_metrics.json").write_text(json.dumps(results, indent=2))
print("sha256", hashlib.sha256((EXPORT / "fire_smoke.pt").read_bytes()).hexdigest())
shutil.copytree(EXPORT, Path("/content/drive/MyDrive/site-safe/artifacts") / RUN_NAME, dirs_exist_ok=True)